In [1]:
import json
from pathlib import Path

import pandas as pd

from bcbench.results.base import BaseEvaluationResult, create_result_from_json

result_folder = Path("result")
runs: dict[str, list[BaseEvaluationResult]] = {}
for jsonl_file in result_folder.glob("*.jsonl"):
    runs[jsonl_file.stem] = [create_result_from_json(json.loads(line)) for line in jsonl_file.read_text(encoding="utf-8").splitlines()]

result_df = pd.DataFrame([{"run_id": run_id, **r.model_dump()} for run_id, results in runs.items() for r in results])

print(f"Category: {result_df['category'].iloc[0]}")
print(f"Agent: {result_df['agent_name'].iloc[0]}")
print(f"Model: {result_df['model'].iloc[0]}")
result_df = result_df.drop(columns=["category", "timeout", "error_message", "agent_name", "model"])

print(f"Loaded {len(result_df)} results from {len(runs)} runs")
print(f"Unique instances: {result_df['instance_id'].nunique()}")
result_df.head(n=2)

Category: EvaluationCategory.BUG_FIX
Agent: GitHub Copilot CLI
Model: claude-haiku-4-5
Loaded 440 results from 8 runs
Unique instances: 55


,run_id,instance_id,project,resolved,build,generated_patch,metrics,experiment
0,20104171950,microsoft__BCApps-4699,Shopify,True,True,diff --git a/src/Apps/W1/Shopify/App/src/Produ...,"{'execution_time': 110.779, 'llm_duration': 10...","{'mcp_servers': None, 'custom_instructions': F..."
1,20104171950,microsoft__BCApps-4766,Subscription Billing,False,True,diff --git a/src/Apps/W1/Subscription Billing/...,"{'execution_time': 98.864, 'llm_duration': 94....","{'mcp_servers': None, 'custom_instructions': F..."


In [2]:
from math import sqrt

import plotly.graph_objects as go

run_ids: list[str] = sorted(result_df["run_id"].unique())

cumulative_data = []
for i in range(2, len(run_ids) + 1):
    subset = result_df[result_df["run_id"].isin(run_ids[:i])]
    per_run = subset.groupby("run_id")["resolved"].mean() * 100

    sem = per_run.std(ddof=1) / sqrt(len(per_run))
    cumulative_data.append({"n_runs": i, "sem": sem})

sem_df = pd.DataFrame(cumulative_data)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=sem_df["n_runs"],
        y=sem_df["sem"],
        mode="lines+markers",
        name="SEM",
        line={"color": "blue", "width": 2},
        marker={"size": 10},
    )
)
fig.update_layout(
    title="SEM Convergence for % Resolved",
    xaxis_title="Number of Runs",
    yaxis_title="Standard Error of Mean (%)",
    xaxis={"tickmode": "linear", "tick0": 2, "dtick": 1},
    yaxis={"rangemode": "tozero"},
    template="plotly_white",
)
fig.show()

In [3]:
per_run = result_df.groupby("run_id")["resolved"].mean() * 100
mean: float = per_run.mean()
sem: float = per_run.std(ddof=1) / sqrt(len(per_run))

print(f"Mean resolution rate: {mean:.2f}%")
print(f"SEM: {sem:.2f}%")
print(f"95% CI: [{mean - 1.96 * sem:.2f}%, {mean + 1.96 * sem:.2f}%]")

Mean resolution rate: 45.45%
SEM: 1.46%
95% CI: [42.60%, 48.31%]
